# Reinforcement Learning From AI Feedback (RLAIF) with Serverless customization on SageMaker AI

## Lab 3 - Fine-Tuning

In this notebook, we are going to fine-tune Qwen 2.5 - 7B Instruct model, using the dataset and reward model set up in the previous labs

## Prerequisites

In [15]:
import os
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

PROFILE_NAME = "njourdan-Admin"

os.environ['AWS_PROFILE'] = PROFILE_NAME
os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'

boto_session = boto3.Session(profile_name=PROFILE_NAME, region_name='us-west-2')
sagemaker_session = Session(boto_session=boto_session)

role = get_execution_role(sagemaker_session=sagemaker_session)

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sagemaker_session.default_bucket()}")
print(f"sagemaker session region: {sagemaker_session.boto_region_name}")

sagemaker role arn: arn:aws:iam::157476309474:role/Admin
sagemaker bucket: sagemaker-us-west-2-157476309474
sagemaker session region: us-west-2


## Set up the model package group

A ModelPackageGroup is a group of versioned models in the SageMaker Model Registry. You can view your ModelPackageGroups in SageMaker AI Studio when selecting Models in the left navigation bar and choosing the "My models" tab.

In [5]:
default_bucket_name = sagemaker_session.default_bucket()
s3_output_path = "s3://{0}/humanlike-qwen25-7b-rlaif/".format(default_bucket_name)

model_package_group_name="demo-humanlike-qwen25-7b-rlaif"
model_package_group_description=""

operation_input_args = {
    "ModelPackageGroupName": model_package_group_name,
    "ModelPackageGroupDescription": model_package_group_description
}

sagemaker_session.sagemaker_client.create_model_package_group(**operation_input_args)

{'ModelPackageGroupArn': 'arn:aws:sagemaker:us-west-2:157476309474:model-package-group/demo-humanlike-qwen25-7b-rlaif',
 'ResponseMetadata': {'RequestId': '925b89d0-dc15-4c1e-853f-203bca45d3ce',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '925b89d0-dc15-4c1e-853f-203bca45d3ce',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '118',
   'date': 'Thu, 22 Jan 2026 11:12:24 GMT'},
  'RetryAttempts': 0}}

## Set up the RLAIFTrainer

In [18]:
from sagemaker.train.rlaif_trainer import RLAIFTrainer
from sagemaker.ai_registry.dataset import DataSet

training_dataset = DataSet.get(name="humanlike-rlaif-train")
val_dataset = DataSet.get(name="humanlike-rlaif-eval")

base_model_id = "huggingface-llm-qwen2-5-7b-instruct"
reward_model_id = "openai.gpt-oss-120b-1:0"
reward_prompt_arn = "arn:aws:sagemaker:us-west-2:157476309474:hub-content/95RJSH51FT2C90GJTEIU6QDJ727VKGNOHE0ITNMJ7FJH1CFCA50G/JsonDoc/humanlike-reward-prompt/2.0.0"

trainer = RLAIFTrainer(
    model=base_model_id,
    model_package_group=model_package_group_name,
    reward_model_id=reward_model_id,
    reward_prompt=reward_prompt_arn,
    training_dataset=training_dataset,
    s3_output_path=s3_output_path,
    sagemaker_session=sagemaker_session,
    role=role,
    accept_eula=True
)

[01/22/26 12:32:54] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=481147;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=39484;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/core/telemetry/telemetry_logging.py#92\92]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

                    INFO     SageMaker session not provided. Using default Session.                  ]8;id=535276;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=831143;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/train/defaults.py#61\61]8;;\

[01/22/26 12:32:56] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=998979;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=112067;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/core/telemetry/telemetry_logging.py#92\92]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

                    INFO     SageMaker session not provided. Using default Session.                  ]8;id=900491;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=309647;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/train/defaults.py#61\61]8;;\

In [19]:
from rich.pretty import pprint

pprint(trainer.hyperparameters.to_dict())

{
│   'global_batch_size': '128',
│   'judge_model_id': 'bedrock/openai.gpt-oss-120b-1:0',
│   'learning_rate': '1e-05',
│   'max_epochs': '2',
│   'max_prompt_length': '1024',
│   'mlflow_run_id': '',
│   'mlflow_tracking_uri': '',
│   'model_name_or_path': 'Qwen/Qwen2.5-7B-Instruct',
│   'name': 'example-name-aajk1',
│   'results_directory': '',
│   'resume_from_path': '',
│   'rollout': '8',
│   'train_val_split_ratio': '0.9'
}

In [20]:
training_job = trainer.train(wait=False)
print(f"Created training job: {training_job.training_job_name}")

[01/22/26 12:33:02] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=150794;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=361927;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/core/telemetry/telemetry_logging.py#92\92]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

                    INFO     Training Job Name:                                                ]8;id=897372;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/train/rlaif_trainer.py\rlaif_trainer.py]8;;\:]8;id=128244;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/train/rlaif_trainer.py#219\219]8;;\
                             huggingface-llm-qwen2-5-7b-instruct-rlaif-20260122123302                              

[01/22/26 12:33:04] INFO     Using first available ready MLflow app:                          ]8;id=135407;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=808150;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/train/common_utils/finetune_utils.py#136\136]8;;\
                             arn:aws:sagemaker:us-west-2:157476309474:mlflow-app/app-VONSKM56                      
                             B7XS                                                                                  

                    INFO     MLflow resource ARN:                                             ]8;id=770885;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=185831;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/train/common_utils/finetune_utils.py#586\586]8;;\
                             arn:aws:sagemaker:us-west-2:157476309474:mlflow-app/app-VONSKM56                      
                             B7XS                                                                                  

                    INFO     Creating training_job resource.                                     ]8;id=564472;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=801368;file:///Users/njourdan/workspace/serverless-customization-workshop/.venv/lib/python3.13/site-packages/sagemaker/core/resources.py#35539\35539]8;;\

Created training job: huggingface-llm-qwen2-5-7b-instruct-rlaif-20260122123302
